# 1. IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.rcParams["figure.figsize"] = (10,6)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# 2. READ DATASET

# to check dataset place    

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#df_0 = pd.read_csv("/kaggle/input/nslkdd/KDDTrain+.txt") 
#df_0 = pd.read_csv("/kaggle/input/nsl-kdd/KDDTrain+.txt")
df_0 = pd.read_csv("/kaggle/input/unsw-nb15-dataset/UNSW-NB15_c/UNSW_NB15_testing-set.csv")
df= df_0.copy()
df.head()

df.drop(columns=['id'], inplace=True, errors='ignore')


# 2.1 ADJUST COLUMNS

columns = (['duration'
,'protocol_type'
,'service'
,'flag'
,'src_bytes'
,'dst_bytes'
,'land'
,'wrong_fragment'
,'urgent'
,'hot'
,'num_failed_logins'
,'logged_in'
,'num_compromised'
,'root_shell'
,'su_attempted'
,'num_root'
,'num_file_creations'
,'num_shells'
,'num_access_files'
,'num_outbound_cmds'
,'is_host_login'
,'is_guest_login'
,'count'
,'srv_count'
,'serror_rate'
,'srv_serror_rate'
,'rerror_rate'
,'srv_rerror_rate'
,'same_srv_rate'
,'diff_srv_rate'
,'srv_diff_host_rate'
,'dst_host_count'
,'dst_host_srv_count'
,'dst_host_same_srv_rate'
,'dst_host_diff_srv_rate'
,'dst_host_same_src_port_rate'
,'dst_host_srv_diff_host_rate'
,'dst_host_serror_rate'
,'dst_host_srv_serror_rate'
,'dst_host_rerror_rate'
,'dst_host_srv_rerror_rate'
,'attack'
,'level'])

df.columns = columns

In [ ]:
df.head(5)

# 2.2 INSIGHTS

In [ ]:
df.info()

In [ ]:
df.describe().T

# 3. DATA CLEANING

# 3.1 NULL VALUES

In [ ]:
df.isnull().sum()

Dataset doesn't contain any null value

In [ ]:
#helper function for deeper analysis
def unique_values(df, columns):
    """Prints unique values and their counts for specific columns in the DataFrame."""

    for column_name in columns:
        print(f"Column: {column_name}\n{'-'*30}")
        unique_vals = df[column_name].unique()
        value_counts = df[column_name].value_counts()
        print(f"Unique Values ({len(unique_vals)}): {unique_vals}\n")
        print(f"Value Counts:\n{value_counts}\n{'='*40}\n")

# 3.2 DUPLICATES

In [ ]:
df.duplicated().sum()

Dataset doesn't contain any duplicated row

# 3.3 OUTLIERS

In [ ]:
df.shape

In [ ]:
plt.figure(figsize=(20, 40))
df.plot(kind='box', subplots=True, layout=(8, 5), figsize=(20, 40))
plt.show()

# 3.4 CLASSIFY ATTACK OR NOT

In [ ]:
#attack_n = []
#for i in df.attack :
#  if i == 'normal':
#    attack_n.append("normal")
#  else:
#    attack_n.append("attack")
#df['attack'] = attack_n 

df['attack'] = df['label'].map({0: 'normal', 1: 'attack'})


In [ ]:
df['attack'].unique()

# 4.2 Service used general

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 8))  # Adjusted figure size
ax = sns.countplot(x='service', data=df)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")  # Rotated labels
plt.xlabel('Service')
plt.ylabel('Count')
plt.title('Count of Services')
plt.grid(True)
plt.show()


# 4.3 Service used effect on attacks

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 8))  # Adjusted figure size
ax = sns.countplot(x='service', hue='attack', data=df)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")  # Rotated labels
plt.xlabel('Service')
plt.ylabel('Count')
plt.title('Distribution of Attacks by Service')
plt.legend(title='Attack Type')
plt.grid(True)
plt.show()


# 5. PREPROCESSING

# 5.1 ENCODING

In [ ]:
cat_features = df.select_dtypes(include='object').columns
cat_features

In [ ]:
from sklearn import preprocessing
#le=preprocessing.LabelEncoder()
#clm=['protocol_type', 'service', 'flag', 'attack']
#for x in clm:
#    df[x]=le.fit_transform(df[x])


from sklearn.preprocessing import LabelEncoder

cat_cols = ['proto', 'service', 'state', 'attack_cat']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# 5.2 TRAIN-TEST-SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

# X = df.drop(["attack"], axis=1)
# y = df["attack"]

X = df.drop(columns=['label', 'attack'])
y = df['label']

# X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.1,random_state=43) 


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
train_index = X_train.columns
train_index

# 5.3 Feature Engineering

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mutual_info = mutual_info_classif(X_train, y_train)
mutual_info = pd.Series(mutual_info)
mutual_info.index = train_index
mutual_info.sort_values(ascending=False)

In [ ]:
mutual_info.sort_values(ascending=False).plot.bar(figsize=(20, 5));

# 5.4 Feature Selection

In [ ]:
from sklearn.feature_selection import SelectKBest
Select_features = SelectKBest(mutual_info_classif, k=30)
Select_features.fit(X_train, y_train)
train_index[Select_features.get_support()]

In [ ]:
#columns=['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
#       'dst_bytes', 'wrong_fragment', 'hot', 'logged_in', 'num_compromised',
#       'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate']

columns=['dur', 'proto', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
       'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt',
       'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
       'ct_src_ltm', 'ct_srv_dst']
#We will continue our model with top 15 features, because dataset is big enough

X_train=X_train[columns]
X_test=X_test[columns]

# 5.5 Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) # we use only transform in order to prevent data leakage





In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load and Preprocess Data
# -----------------------------

#def load_nsl_kdd(file_path):
#    column_names = [f'feature_{i}' for i in range(41)] + ['label']
#    df = pd.read_csv(file_path, names=column_names)
#    return df

#def preprocess_data(df):
#    # Convert attack labels to binary: normal (0), attack (1)
#    df['label'] = df['label'].apply(lambda x: 0 if 'normal' in x else 1)

#    # Encode categorical features
#    for col in df.select_dtypes(include=['object']).columns:
#        df[col] = LabelEncoder().fit_transform(df[col])

#    # Normalize features
#    features = df.drop('label', axis=1)
#    scaler = MinMaxScaler()
#    features = scaler.fit_transform(features)

#    labels = df['label'].values
#    return features, labels, scaler

# Load dataset
#df = load_nsl_kdd('KDDTrain+.txt')
#X, y, scaler = preprocess_data(df)

# Extract only attack samples for GAN training
X_attack = X_train   #x[y == 1]
#X_attack = torch.tensor(X_attack, dtype=torch.float32)

# --------------------------------
# 2. Define GAN Architecture
# --------------------------------

class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# --------------------------------
# 3. Train GAN
# --------------------------------

def train_gan(generator, discriminator, data, noise_dim, num_epochs=10, batch_size=64):       #num_epochs=100
    criterion = nn.BCELoss()
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0002)
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0002)

    for epoch in range(num_epochs):
        for i in range(0, len(data), batch_size):
            #real_data = data[i:i+batch_size]

            real_data = torch.tensor(data[i:i+batch_size], dtype=torch.float32)
            batch_size = 64 #real_data.size(0)

            # Train Discriminator
            noise = torch.randn(batch_size, noise_dim)
            fake_data = generator(noise)

            d_real = discriminator(real_data)
            d_fake = discriminator(fake_data.detach())

            d_loss = criterion(d_real, torch.ones_like(d_real)) + \
                     criterion(d_fake, torch.zeros_like(d_fake))

            d_optimizer.zero_grad()
            d_loss.backward()
            d_optimizer.step()

            # Train Generator
            noise = torch.randn(batch_size, noise_dim)
            fake_data = generator(noise)
            d_fake = discriminator(fake_data)

            g_loss = criterion(d_fake, torch.ones_like(d_fake))

            g_optimizer.zero_grad()
            g_loss.backward()
            g_optimizer.step()

        if (epoch+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]  D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}")

# --------------------------------
# 4. Generate Synthetic Attack Traffic
# --------------------------------

noise_dim = 32
input_dim = X_attack.shape[1]

G = Generator(noise_dim, input_dim)
D = Discriminator(input_dim)

train_gan(G, D, X_attack, noise_dim)

# Generate synthetic samples
num_samples = 1000
noise = torch.randn(num_samples, noise_dim)
synthetic_attacks = G(noise).detach().numpy()

# Rescale back to original scale
synthetic_attacks_rescaled = scaler.inverse_transform(synthetic_attacks)

# Save to CSV
pd.DataFrame(synthetic_attacks_rescaled).to_csv('synthetic_attacks.csv', index=False)
print("✅ Synthetic attack traffic saved to 'synthetic_attacks.csv'")

# 6. MODEL BUILD

In [ ]:
##------------------ old model
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


In [ ]:
###----------------new model    deep learning
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.utils import to_categorical

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Convert labels to categorical (if you are doing multi-class classification)
# y_train = to_categorical(y_train)
# y_test = to_categorical(y_test)

#----------------------------------this work
#def create_model(learning_rate=0.01):
#    model = Sequential()
#    model.add(Dense(units=64, activation='relu', input_shape=(15,)))  # Replace input_dim
#    model.add(Dense(units=1, activation='sigmoid'))  # Adjust based on your output layer
#    model.compile(loss='binary_crossentropy',
#                  optimizer=Adam(learning_rate=learning_rate),
#                  metrics=['accuracy'])
#    return model
#
#
#model = create_model(learning_rate=0.01)
#
#
##model = build_model(X_train)
#print(X_train.shape[1])    #,print("ytrain=",y_train)
#


In [ ]:
#print("x=", X)    # 15
print("X_train= ",X_train)    #42

In [ ]:
# -----------------------------------       Or    Second method    LSTM
# Your LSTM model expects 3D input of shape

# Ensure your input is 3D


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
import numpy as np

# Reshape input to (samples, timesteps, features)
#X_train = np.array(X_train)
#X_test = np.array(X_test)
#y_train = np.array(y_train)
#

## Convert to numpy
#X_train = np.array(X_train)
#X_test = np.array(X_test)
#y_train = np.array(y_train)

# Sanity check
#print("X_train shape before reshape:", X_train.shape)
#print("X_test shape before reshape:", X_test.shape)

# Define LSTM structure
#timesteps = 15
#features = 3

#timesteps = X_train.shape[1]  # 11
#features = 1

# Total features must match
# assert X_train.shape[1] == timesteps * features, "Feature mismatch!"

# Reshape correctly
#X_train = X_train.reshape(X_train.shape[0], timesteps, features)
#X_test = X_test.reshape(X_test.shape[0], timesteps, features)

timesteps = X_train.shape[1]   # 29
features = 1

X_train = X_train.reshape(X_train.shape[0], timesteps, features)
X_test  = X_test.reshape(X_test.shape[0], timesteps, features)



# Define model   --------------------------------------------------
def create_lstm_model(learning_rate=0.01):
    model = Sequential()
    model.add(LSTM(units=64, input_shape=(timesteps, features)))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model

# Train model
#model2 = create_lstm_model()
#model2.fit(X_train, y_train[:len(X_train)], epochs=10, batch_size=128, validation_split=0.2)

model2 = create_lstm_model()
model2.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)


# Predict
proba_normal = model2.predict(X_test).flatten()

# Compute neutrosophic values
proba_attack = 1 - proba_normal
I = 1 - np.abs(2 * proba_normal - 1)
neutrosophic_values = np.stack([proba_normal, I, proba_attack], axis=1)

# Show result
print("Neutrosophic outputs (T, I, F):")
print(neutrosophic_values[:5])


# ----------------------------
# Defer Indeterminate Cases
# ----------------------------

indeterminacy_threshold = 0.4  # adjust as needed
final_preds = []

for t, i, f in neutrosophic_values:
    if i >= indeterminacy_threshold:
        final_preds.append(-1)  # Defer decision
    else:
        final_preds.append(1 if t >= 0.5 else 0)

final_preds = np.array(final_preds)

# ----------------------------
# Evaluate Model (Excluding Deferred Cases)
# ----------------------------

# Only evaluate where predictions were made
mask = final_preds != -1
num_deferred = np.sum(~mask)
accuracy = np.mean(final_preds[mask] == y_test[mask])

print(f"\nDeferred decisions: {num_deferred}/{len(y_test)}")
print(f"Accuracy (excluding deferred): {accuracy:.4f}")



In [ ]:
4067888 / 11

In [ ]:
##  step 3       this part  give accuracy 99.9  ~ 100 %

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

# ----------------------------
# Assume: X_train, X_test are 3D numpy arrays of shape (samples, 15, 3)
# y_train, y_test are binary labels (0 = benign, 1 = attack)
# Each timestep's feature = (T, I, F)
# ----------------------------

# Example placeholders — replace with your actual data
# X_train = np.random.rand(10000, 15, 3)
# y_train = np.random.randint(0, 2, size=(10000,))
# X_test = np.random.rand(3000, 15, 3)
# y_test = np.random.randint(0, 2, size=(3000,))

# ----------------------------
# Define LSTM Model
# ----------------------------

def create_lstm_model(learning_rate=0.001):
    model = Sequential()
    model.add(LSTM(units=64, input_shape=(15, 3)))  # 15 time steps, 3 features (T, I, F)
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model

# ----------------------------
# Train Model
# ----------------------------

model = create_lstm_model()

history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.2,
                    verbose=1)

# ----------------------------
# Predict and Compute Neutrosophic Values
# ----------------------------

proba_normal = model.predict(X_test).flatten()      # Truth (T)
proba_attack = 1 - proba_normal                     # Falsity (F)
I = 1 - np.abs(2 * proba_normal - 1)                # Indeterminacy (I)
neutrosophic_values = np.stack([proba_normal, I, proba_attack], axis=1)

print("\nSample Neutrosophic outputs (T, I, F):")
print(neutrosophic_values[:5])

# ----------------------------
# Handle Indeterminate Cases by Deferring
# ----------------------------

indeterminacy_threshold = 0.4  # Adjust based on confidence needs

final_preds = []
for t, i, f in neutrosophic_values:
    if i >= indeterminacy_threshold:
        final_preds.append(-1)  # Defer decision
    else:
        final_preds.append(1 if t >= 0.5 else 0)

final_preds = np.array(final_preds)

# ----------------------------
# Evaluate (Exclude Deferred)
# ----------------------------

mask = final_preds != -1
num_deferred = np.sum(~mask)
accuracy = np.mean(final_preds[mask] == y_test[mask])

print(f"\nDeferred decisions: {num_deferred}/{len(y_test)}")
print(f"Accuracy (excluding deferred): {accuracy:.4f}")

# Optional: Classification report
print("\nClassification Report (excluding deferred cases):")
print(classification_report(y_test[mask], final_preds[mask]))

In [ ]:
#the same code like prievous but with plot



import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

# ----------------------------
# Assume: X_train, X_test, y_train, y_test are preprocessed
# Shape: X = (samples, 15, 3), y = (samples,)
# ----------------------------

# ----------------------------
# Define LSTM Model
# ----------------------------

def create_lstm_model(learning_rate=0.001):
    model = Sequential()
    model.add(LSTM(units=64, input_shape=(15, 3)))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model

# ----------------------------
# Train Model
# ----------------------------

model = create_lstm_model()

history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.2,
                    verbose=1)

# ----------------------------
# Plot Accuracy and Loss
# ----------------------------

plt.figure(figsize=(12, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ----------------------------
# Predict and Compute Neutrosophic Values
# ----------------------------

proba_normal = model.predict(X_test).flatten()
proba_attack = 1 - proba_normal
I = 1 - np.abs(2 * proba_normal - 1)
neutrosophic_values = np.stack([proba_normal, I, proba_attack], axis=1)

print("\nSample Neutrosophic outputs (T, I, F):")
print(neutrosophic_values[:5])

# ----------------------------
# Handle Indeterminate Cases by Deferring
# ----------------------------

indeterminacy_threshold = 0.4

final_preds = []
for t, i, f in neutrosophic_values:
    if i >= indeterminacy_threshold:
        final_preds.append(-1)
    else:
        final_preds.append(1 if t >= 0.5 else 0)

final_preds = np.array(final_preds)

# ----------------------------
# Evaluate (Exclude Deferred)
# ----------------------------

mask = final_preds != -1
num_deferred = np.sum(~mask)
accuracy = np.mean(final_preds[mask] == y_test[mask])

print(f"\nDeferred decisions: {num_deferred}/{len(y_test)}")
print(f"Accuracy (excluding deferred): {accuracy:.4f}")

print("\nClassification Report (excluding deferred cases):")
print(classification_report(y_test[mask], final_preds[mask]))

In [ ]:
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

# ----------------------------
# Environment Constants
# ----------------------------
STATE_SIZE = 3  # T, I, F
ACTION_SIZE = 3  # 0: Block, 1: Allow, 2: Inspect

# ----------------------------
# Reward Function
# ----------------------------
def get_reward(true_label, action):
    if action == 2:  # Inspect
        return 0  # Neutral reward
    elif action == true_label:  # Correct decision
        return 1
    else:  # Wrong classification
        return -1

# ----------------------------
# DQN Agent Definition
# ----------------------------
class DQNAgent:
    def __init__(self):
        self.memory = deque(maxlen=5000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.learning_rate = 0.001
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential()
        model.add(Input(shape=(STATE_SIZE,)))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(ACTION_SIZE, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(ACTION_SIZE)
        q_values = self.model.predict(state[np.newaxis], verbose=0)
        return np.argmax(q_values[0])

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size=64):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            q_update = reward
            if not done:
                q_update += self.gamma * np.amax(self.model.predict(next_state[np.newaxis], verbose=0)[0])
            q_values = self.model.predict(state[np.newaxis], verbose=0)
            q_values[0][action] = q_update
            self.model.fit(state[np.newaxis], q_values, verbose=0)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# ----------------------------
# LSTM Model to Learn Neutrosophic Features
# ----------------------------
def create_lstm_model(learning_rate=0.001):
    model = Sequential()
    model.add(LSTM(units=64, input_shape=(15, 3)))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy',
                  optimizer=Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model

# ----------------------------
# Train LSTM
# ----------------------------
model = create_lstm_model()
history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=128,
                    validation_split=0.2,
                    verbose=1)

# ----------------------------
# Plot Accuracy and Loss
# ----------------------------
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ----------------------------
# Predict and Compute Neutrosophic Values
# ----------------------------
proba_normal = model.predict(X_test).flatten()
proba_attack = 1 - proba_normal
indeterminacy = 1 - np.abs(2 * proba_normal - 1)
neutrosophic_states = np.stack([proba_normal, indeterminacy, proba_attack], axis=1)

# ----------------------------
# Train DQN on Neutrosophic Features
# ----------------------------
agent = DQNAgent()
EPISODES = 50
y_test = y_test.reset_index(drop=True)

for episode in range(EPISODES):
    total_reward = 0
    for i in range(len(neutrosophic_states)):
        state = neutrosophic_states[i]
        true_label = y_test[i]

        action = agent.act(state)
        next_state = neutrosophic_states[i + 1] if i + 1 < len(neutrosophic_states) else state
        reward = get_reward(true_label, action)
        done = i == len(neutrosophic_states) - 1

        agent.remember(state, action, reward, next_state, done)
        total_reward += reward

    agent.replay()
    print(f"Episode {episode+1}/{EPISODES}, Total Reward: {total_reward}, Epsilon: {agent.epsilon:.3f}")

# ----------------------------
# Final Evaluation
# ----------------------------
final_actions = []
true_labels = []

for i in range(len(neutrosophic_states)):
    state = neutrosophic_states[i]
    action = np.argmax(agent.model.predict(state[np.newaxis], verbose=0))
    final_actions.append(action)
    true_labels.append(y_test[i])

mapped_preds = []
mapped_labels = []

for a, y in zip(final_actions, true_labels):
    if a == 2:
        continue  # Skip inspect cases
    mapped_preds.append(1 if a == 0 else 0)  # 0 → block → attack, 1 → allow → benign
    mapped_labels.append(y)

print("\nClassification Report (excluding 'inspect'):")
print(classification_report(mapped_labels, mapped_preds))
print(f"Inspect cases: {final_actions.count(2)} / {len(final_actions)}")

In [ ]:
#plot ROC curve 

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, proba_attack)
roc_auc = auc(fpr, tpr)

# Plot ROC
plt.figure()
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Intrusion Detection System')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import random
from collections import deque
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

# Environment Constants
STATE_SIZE = 3  # T, I, F
ACTION_SIZE = 3  # 0: Block, 1: Allow, 2: Inspect

# Reward Scheme
def get_reward(true_label, action):
    if action == 2:  # Inspect
        return 0  # No penalty
    elif action == true_label:  # Correct classification
        return 1
    else:
        return -1  # Wrong classification

# DQN Agent
class DQNAgent:
    def __init__(self):
        self.memory = deque(maxlen=5000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.learning_rate = 0.001
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential()
        model.add(Input(shape=(STATE_SIZE,)))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(64, activation='relu'))
        model.add(Dense(ACTION_SIZE, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(ACTION_SIZE)
        q_values = self.model.predict(state[np.newaxis], verbose=0)
        return np.argmax(q_values[0])

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size=64):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            q_update = reward
            if not done:
                q_update = reward + self.gamma * np.amax(self.model.predict(next_state[np.newaxis], verbose=0)[0])
            q_values = self.model.predict(state[np.newaxis], verbose=0)
            q_values[0][action] = q_update
            self.model.fit(state[np.newaxis], q_values, verbose=0)
        
        # Reduce exploration over time
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# ----------------------------
# Use LSTM to Extract T, I, F
# ----------------------------

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam


def create_lstm_model():
    model = Sequential()
    model.add(LSTM(units=64, input_shape=(15, 3)))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    return model

lstm_model = create_lstm_model()
lstm_model.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.2)

proba_normal = lstm_model.predict(X_test).flatten()
proba_attack = 1 - proba_normal
indeterminacy = 1 - np.abs(2 * proba_normal - 1)
neutrosophic_states = np.stack([proba_normal, indeterminacy, proba_attack], axis=1)

# ----------------------------
# Train DQN Agent
# ----------------------------

agent = DQNAgent()
EPISODES = 2    #10

y_test = y_test.reset_index(drop=True)
#y_test_array = y_test.to_numpy()  # Add this line before the loop

for episode in range(EPISODES):
    total_reward = 0
    for i in range(len(neutrosophic_states)):
        state = neutrosophic_states[i]
        true_label = y_test[i]
        #true_label = y_test_array[i]  # Use NumPy indexing

        action = agent.act(state)
        next_state = neutrosophic_states[i + 1] if i + 1 < len(neutrosophic_states) else state
        reward = get_reward(true_label, action)
        done = i == len(neutrosophic_states) - 1

        agent.remember(state, action, reward, next_state, done)
        total_reward += reward

    agent.replay()
    print(f"Episode {episode+1}/{EPISODES}, Total Reward: {total_reward}, Epsilon: {agent.epsilon:.3f}")

# ----------------------------
# Final Evaluation
# ----------------------------

final_actions = []
true_labels = []


for i in range(len(neutrosophic_states)):
    state = neutrosophic_states[i]
    action = np.argmax(agent.model.predict(state[np.newaxis], verbose=0))
    final_actions.append(action)
    true_labels.append(y_test[i])

# Evaluate: map actions 0 (Block) as class 1, 1 (Allow) as class 0; exclude 2 (Inspect)
mapped_preds = []
mapped_labels = []

for a, y in zip(final_actions, true_labels):
    if a == 2:
        continue  # skip inspect cases
    mapped_preds.append(1 if a == 0 else 0)
    mapped_labels.append(y)

print("\nClassification Report (excluding 'inspect'):")
print(classification_report(mapped_labels, mapped_preds))

print(f"Inspect cases: {final_actions.count(2)} / {len(final_actions)}")